# PawInHand Evidence Audit

## 목적

이 노트북은 PawInHand 3개월 수집 데이터에서 **상세 페이지의 `personality.comment`가 추천 feature의 근거로 실제 어느 정도 쓸 수 있는지**를 별도로 확인합니다.

기존 audit의 전체 coverage 분석은 유지하되, 이번 버전은 다음 질문을 추가로 확인합니다.

1. `personality.comment`는 얼마나 존재하는가?
2. 개/고양이, 보호중/종료 상태에 따라 기록 편차가 큰가?
3. comment는 단순 홍보 문구인지, 아니면 `사람 친화력`, `타 동물 친화력`, `에너지`, `좋아하는 것`, `싫어하는 것`, `이상적인 가정`처럼 추천에 연결할 수 있는 내용을 반복적으로 포함하는가?
4. comment는 `personality.tags`, gauge, `special_mark`가 이미 주는 정보에 **추가 evidence**를 제공하는가?
5. comment가 특정 기관/템플릿/프로그램에 편중되어 있는가?
6. raw JSON 안에 작성 주체·보호센터·source를 추적할 수 있는 필드가 있는가?

### 해석 제한

- `personality.comment`가 PawInHand 화면에서 수집된 상세정보라고 해서 **PawInHand 직원이 직접 작성한 텍스트라고 단정하지 않습니다.**
- comment 본문에 보호센터명, 입양문의, PawInHand 등의 표현이 있는지 확인할 수는 있지만, **텍스트만으로 실제 작성자(author)를 확정할 수 없습니다.**
- keyword hit는 evidence 후보를 찾기 위한 screening일 뿐 실제 feature label이나 evidence coverage가 아닙니다.
- 이번 notebook에서 새로 만드는 수동검토 칼럼은 `manual_*` 이름을 사용해 원본 필드와 구분합니다.

In [30]:
from pathlib import Path
from collections import Counter
import json
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 80)

cwd = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "analysis").is_dir() and (p / "results").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("저장소 루트 또는 analysis/ 아래에서 노트북을 실행해 주세요.")

OUTPUT_DIR = PROJECT_ROOT / "results/pawinhand/evidence_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_dirs = [
    PROJECT_ROOT / "data/pawinhand/raw/three_months",
    PROJECT_ROOT / "pawinhand-crawler/data/three_months",
]

DATA_DIR = next((p for p in candidate_dirs if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "PawInHand 3개월 데이터 폴더를 찾지 못했습니다. "
        "data/pawinhand/raw/three_months 또는 "
        "pawinhand-crawler/data/three_months 경로를 확인하세요."
    )

files = sorted(DATA_DIR.glob("animals-*.json"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("JSON files:", len(files))

PROJECT_ROOT: c:\Users\USER\Desktop\college\2026\4-2_lecture\캡스톤\bigdata_capstone
DATA_DIR: c:\Users\USER\Desktop\college\2026\4-2_lecture\캡스톤\bigdata_capstone\pawinhand-crawler\data\three_months
OUTPUT_DIR: c:\Users\USER\Desktop\college\2026\4-2_lecture\캡스톤\bigdata_capstone\results\pawinhand\evidence_audit
JSON files: 23


In [31]:
records = []

for path in files:
    data = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(data, list):
        records.extend(data)
    elif isinstance(data, dict) and isinstance(data.get("data"), list):
        records.extend(data["data"])
    else:
        raise ValueError(f"예상하지 못한 JSON 구조: {path}")

notice_ids = [r.get("notice_id") for r in records]

sanity = pd.DataFrame({
    "metric": [
        "JSON 파일 수",
        "전체 record 수",
        "notice_id 결측",
        "notice_id 중복",
    ],
    "value": [
        len(files),
        len(records),
        sum(x in (None, "") for x in notice_ids),
        len(notice_ids) - len(set(notice_ids)),
    ],
})

display(sanity)

,metric,value
0,JSON 파일 수,23
1,전체 record 수,22511
2,notice_id 결측,0
3,notice_id 중복,0


## 1. 원본 schema 확인

먼저 `personality.comment`가 어떤 nested object에서 오는지 확인하고, raw JSON에 보호센터·기관·source·작성주체를 추적할 만한 필드가 있는지도 탐색합니다.

이 단계는 **필드명을 추측해서 결론내리지 않기 위한 schema audit**입니다.

In [32]:
def walk_paths(obj, prefix=""):
    paths = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            paths.append(path)
            paths.extend(walk_paths(value, path))

    elif isinstance(obj, list) and obj:
        paths.extend(walk_paths(obj[0], f"{prefix}[]"))

    return paths


path_counts = Counter()

for animal in records:
    for path in set(walk_paths(animal)):
        path_counts[path] += 1

schema_summary = pd.DataFrame(
    [
        {"path": path, "record_n": n, "record_pct": round(n / len(records) * 100, 2)}
        for path, n in path_counts.items()
    ]
).sort_values(["record_n", "path"], ascending=[False, True])

print("상위 schema path")
display(schema_summary.head(80))

source_pattern = re.compile(
    r"care|shelter|center|centre|organization|organisation|org|source|agency|author|"
    r"보호|센터|기관|작성|출처",
    flags=re.I,
)

source_schema_candidates = schema_summary[
    schema_summary["path"].str.contains(source_pattern, regex=True, na=False)
].copy()

print("\n작성주체/보호센터/source 후보 path")
display(source_schema_candidates.head(100))

상위 schema path


,path,record_n,record_pct
12,age_raw,22511,100.00
16,animal_name,22511,100.00
8,breed,22511,100.00
5,collected_at,22511,100.00
26,color,22511,100.00
17,detail_status,22511,100.00
28,found_location,22511,100.00
13,health,22511,100.00
3,health.comment,22511,100.00
7,health.tests,22511,100.00



작성주체/보호센터/source 후보 path


,path,record_n,record_pct
25,shelter_address,22511,100.0
0,shelter_name,22511,100.0
24,shelter_tel,22511,100.0
4,source_url,22511,100.0


> 위 표에 source/기관 후보 필드가 보이면 실제 값을 추가 확인합니다.  
> 후보가 없다고 해서 작성 주체 정보가 원래 없다는 뜻은 아니며, 현재 크롤링 schema에 저장되지 않았을 수도 있습니다.

In [33]:
def present(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, dict):
        return any(present(v) for v in value.values())
    if isinstance(value, list):
        return any(present(v) for v in value)
    return True


def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " | ".join(str(v).strip() for v in value if str(v).strip())
    return str(value).strip()


rows = []

for animal in records:
    personality = animal.get("personality") or {}
    gauges = personality.get("gauges") or {}
    tags = personality.get("tags") or []
    health = animal.get("health") or {}

    special_mark = clean_text(animal.get("special_mark"))
    personality_comment = clean_text(personality.get("comment"))
    personality_tags = clean_text(tags)
    health_comment = clean_text(health.get("comment"))

    rows.append({
        "notice_id": animal.get("notice_id"),
        "species": animal.get("species"),
        "breed": animal.get("breed"),
        "status_raw": animal.get("status_raw"),
        "detail_status_raw": animal.get("detail_status"),
        "shelter_name": animal.get("shelter_name"),
        "special_mark": special_mark,
        "personality_comment": personality_comment,
        "personality_tags": personality_tags,
        "health_comment": health_comment,
        "gauge_건강상태": gauges.get("건강상태"),
        "gauge_활동성": gauges.get("활동성"),
        "gauge_사회성": gauges.get("사회성"),
        "gauge_친화도": gauges.get("친화도"),
    })

df = pd.DataFrame(rows)

def has_real_detail(animal):
    personality = animal.get("personality") or {}
    health = animal.get("health") or {}
    return any(present(v) for v in [
        personality.get("comment"), personality.get("tags"),
        personality.get("gauges"), health.get("comment"), health.get("tests"),
    ])

df["detail_present_clean"] = [has_real_detail(a) for a in records]
df["comment_present"] = df["personality_comment"].map(present)
df["tags_present"] = df["personality_tags"].map(present)
df["health_comment_present"] = df["health_comment"].map(present)
df["any_gauge_present"] = df[
    ["gauge_건강상태", "gauge_활동성", "gauge_사회성", "gauge_친화도"]
].apply(lambda row: any(present(x) for x in row), axis=1)

active_df = df[df["status_raw"] == "보호중"].copy()
comment_df = df[df["comment_present"]].copy()
active_comment_df = active_df[active_df["comment_present"]].copy()

print("shape:", df.shape)
print("comment rows:", len(comment_df))
print("active comment rows:", len(active_comment_df))
print("raw detail present:", int(df["detail_status_raw"].eq("present").sum()))
print("clean detail present:", int(df["detail_present_clean"].sum()))
print("raw false positives:", int((df["detail_status_raw"].eq("present") & ~df["detail_present_clean"]).sum()))
display(df.head(3))

shape: (22511, 19)
comment rows: 381
active comment rows: 141
raw detail present: 843
clean detail present: 841
raw false positives: 2


,notice_id,species,breed,status_raw,detail_status_raw,shelter_name,special_mark,personality_comment,personality_tags,health_comment,gauge_건강상태,gauge_활동성,gauge_사회성,gauge_친화도,detail_present_clean,comment_present,tags_present,health_comment_present,any_gauge_present
0,대구-달성-2026-00410,고양이,[고양이] 한국 고양이,보호중,not_provided,119동물병원,생후 2~3일령 추정,,,,None,None,None,None,False,False,False,False,False
1,대구-달성-2026-00409,고양이,[고양이] 한국 고양이,보호중,not_provided,119동물병원,생후 2~3일령 추정,,,,None,None,None,None,False,False,False,False,False
2,대구-달성-2026-00408,고양이,[고양이] 한국 고양이,보호중,not_provided,119동물병원,생후 2~3일령 추정,,,,None,None,None,None,False,False,False,False,False


## 2. 전체 source coverage와 `personality.comment` 비중

In [34]:
SOURCE_COLUMNS = {
    "special_mark": lambda x: x["special_mark"].map(present),
    "personality_comment": lambda x: x["personality_comment"].map(present),
    "personality_tags": lambda x: x["personality_tags"].map(present),
    "health_comment": lambda x: x["health_comment"].map(present),
    "detail_present_clean": lambda x: x["detail_present_clean"],
    "gauge_활동성": lambda x: x["gauge_활동성"].map(present),
    "gauge_사회성": lambda x: x["gauge_사회성"].map(present),
    "gauge_친화도": lambda x: x["gauge_친화도"].map(present),
    "gauge_건강상태": lambda x: x["gauge_건강상태"].map(present),
}

coverage_rows = []

for scope, scope_df in [("all_3months", df), ("active_only", active_df)]:
    for source, make_mask in SOURCE_COLUMNS.items():
        mask = make_mask(scope_df)
        coverage_rows.append({
            "scope": scope,
            "source": source,
            "n_total": len(scope_df),
            "non_null_n": int(mask.sum()),
            "coverage_pct": round(mask.mean() * 100, 2),
        })

coverage_summary = pd.DataFrame(coverage_rows)
display(coverage_summary)

,scope,source,n_total,non_null_n,coverage_pct
0,all_3months,special_mark,22511,22511,100.00
1,all_3months,personality_comment,22511,381,1.69
2,all_3months,personality_tags,22511,576,2.56
3,all_3months,health_comment,22511,604,2.68
4,all_3months,detail_present_clean,22511,841,3.74
5,all_3months,gauge_활동성,22511,815,3.62
6,all_3months,gauge_사회성,22511,815,3.62
7,all_3months,gauge_친화도,22511,815,3.62
8,all_3months,gauge_건강상태,22511,812,3.61
9,active_only,special_mark,7791,7791,100.00


## 3. `personality.comment`의 종별·상태별 분포

In [35]:
def coverage_by(group_cols):
    grouped = (
        df.groupby(group_cols, dropna=False)
          .agg(
              n=("notice_id", "size"),
              comment_n=("comment_present", "sum"),
              tags_n=("tags_present", "sum"),
              gauge_n=("any_gauge_present", "sum"),
          )
          .reset_index()
    )

    grouped["comment_pct"] = (grouped["comment_n"] / grouped["n"] * 100).round(2)
    grouped["tags_pct"] = (grouped["tags_n"] / grouped["n"] * 100).round(2)
    grouped["gauge_pct"] = (grouped["gauge_n"] / grouped["n"] * 100).round(2)

    return grouped


comment_by_species = coverage_by(["species"])
comment_by_status = coverage_by(["status_raw"])
comment_by_species_status = coverage_by(["species", "status_raw"])

print("[species]")
display(comment_by_species)

print("\n[status]")
display(comment_by_status)

print("\n[species x status]")
display(comment_by_species_status)

[species]


,species,n,comment_n,tags_n,gauge_n,comment_pct,tags_pct,gauge_pct
0,개,13786,344,543,785,2.50,3.94,5.69
1,고양이,8190,36,32,37,0.44,0.39,0.45
2,기타,535,1,1,1,0.19,0.19,0.19



[status]


,status_raw,n,comment_n,tags_n,gauge_n,comment_pct,tags_pct,gauge_pct
0,보호중,7791,141,271,321,1.81,3.48,4.12
1,종료(기증),828,9,35,48,1.09,4.23,5.80
2,종료(반환),2575,12,14,37,0.47,0.54,1.44
3,종료(방사),459,0,0,0,0.00,0.00,0.00
4,종료(안락사),1613,30,58,162,1.86,3.60,10.04
5,종료(입양),4240,184,175,225,4.34,4.13,5.31
6,종료(자연사),5005,5,23,30,0.10,0.46,0.60



[species x status]


,species,status_raw,n,comment_n,tags_n,gauge_n,comment_pct,tags_pct,gauge_pct
0,개,보호중,5660,125,258,306,2.21,4.56,5.41
1,개,종료(기증),627,9,35,48,1.44,5.58,7.66
2,개,종료(반환),2442,12,14,37,0.49,0.57,1.52
3,개,종료(안락사),1329,30,58,162,2.26,4.36,12.19
4,개,종료(입양),2478,164,156,203,6.62,6.30,8.19
5,개,종료(자연사),1250,4,22,29,0.32,1.76,2.32
6,고양이,보호중,1998,15,13,15,0.75,0.65,0.75
7,고양이,종료(기증),176,0,0,0,0.00,0.00,0.00
8,고양이,종료(반환),87,0,0,0,0.00,0.00,0.00
9,고양이,종료(방사),452,0,0,0,0.00,0.00,0.00


## 4. Comment 길이와 기록 밀도

단순히 comment가 존재하는지만 보지 않고, 실제 정보량 차이가 큰지 확인합니다.

길이가 길다고 곧바로 품질이 높다는 뜻은 아니지만, `한두 단어 수준`과 `행동·환경을 여러 문장으로 설명하는 수준`을 구분하는 기본 지표로 사용합니다.

In [36]:
comment_df["comment_char_len"] = comment_df["personality_comment"].str.len()
comment_df["comment_line_n"] = comment_df["personality_comment"].str.count(r"\n") + 1

length_summary = pd.DataFrame({
    "metric": [
        "comment_n",
        "char_mean",
        "char_median",
        "char_p25",
        "char_p75",
        "char_p90",
        "char_max",
        "line_median",
    ],
    "value": [
        len(comment_df),
        round(comment_df["comment_char_len"].mean(), 1),
        round(comment_df["comment_char_len"].median(), 1),
        round(comment_df["comment_char_len"].quantile(0.25), 1),
        round(comment_df["comment_char_len"].quantile(0.75), 1),
        round(comment_df["comment_char_len"].quantile(0.90), 1),
        int(comment_df["comment_char_len"].max()),
        round(comment_df["comment_line_n"].median(), 1),
    ],
})

display(length_summary)

length_bins = pd.cut(
    comment_df["comment_char_len"],
    bins=[0, 50, 150, 300, 700, float("inf")],
    labels=["1-50", "51-150", "151-300", "301-700", "701+"],
    include_lowest=True,
)

comment_length_distribution = (
    length_bins.value_counts(sort=False)
    .rename_axis("char_bin")
    .reset_index(name="n")
)

comment_length_distribution["pct"] = (
    comment_length_distribution["n"] / len(comment_df) * 100
).round(2)

display(comment_length_distribution)

,metric,value
0,comment_n,381.0
1,char_mean,283.1
2,char_median,248.0
3,char_p25,135.0
4,char_p75,383.0
5,char_p90,563.0
6,char_max,1278.0
7,line_median,11.0


,char_bin,n,pct
0,1-50,46,12.07
1,51-150,68,17.85
2,151-300,113,29.66
3,301-700,143,37.53
4,701+,11,2.89


## 5. Comment에 반복되는 준구조화 섹션

`personality.comment`가 단순 자유서술만인지, 아니면 추천에 활용할 수 있는 섹션 구조가 반복되는지 확인합니다.

아래는 **문자열 출현 빈도**이며 의미가 올바르게 기록됐다는 검증은 아닙니다.

In [37]:
SECTION_PATTERNS = {
    "성격_성향": r"성격\s*[·ㆍ/&]?\s*성향|성격\s*[:：]",
    "사람_친화력": r"사람\s*친화력|사람과의\s*(?:관계|상호작용)|사람을\s*(?:좋아|잘\s*따)",
    "타동물_친화력": r"타\s*동물\s*친화력|다른\s*(?:개|강아지|고양이).*(?:잘\s*지|친화|어울)",
    "에너지_활동성": r"에너지\s*(?:레벨|수준)|활동량|활동성",
    "좋아하는것": r"좋아하는\s*것|좋아해요|좋아함",
    "싫어하는것": r"싫어하는\s*것|싫어해요|싫어함",
    "이상적인가정": r"이상적인\s*가정|추천\s*가정|가정\s*추천|단독가정추천",
    "교육_훈련": r"교육\s*필요|훈련|사회화|배변교육",
    "산책_운동": r"산책|운동",
    "혼자있는시간": r"혼자\s*있는\s*시간|분리불안|상주\s*인원",
    "아이_아동": r"어린아이|아이와\s*(?:잘|함께)|아동|유아",
    "고양이경험": r"고양이\s*경험|고양이와|다른\s*고양이",
}

section_rows = []

for name, pattern in SECTION_PATTERNS.items():
    mask = comment_df["personality_comment"].str.contains(
        pattern, regex=True, case=False, na=False
    )

    section_rows.append({
        "section_or_topic": name,
        "hit_n": int(mask.sum()),
        "pct_of_comments": round(mask.mean() * 100, 2),
    })

comment_structure_summary = (
    pd.DataFrame(section_rows)
      .sort_values(["hit_n", "section_or_topic"], ascending=[False, True])
)

display(comment_structure_summary)

,section_or_topic,hit_n,pct_of_comments
1,사람_친화력,105,27.56
4,좋아하는것,87,22.83
3,에너지_활동성,65,17.06
2,타동물_친화력,60,15.75
7,교육_훈련,56,14.70
5,싫어하는것,55,14.44
6,이상적인가정,54,14.17
8,산책_운동,42,11.02
0,성격_성향,31,8.14
9,혼자있는시간,22,5.77


## 6. Comment와 tag/gauge 동시 존재 여부

In [38]:
cooccurrence_items = {
    "comment + tags": comment_df["tags_present"],
    "comment + any_gauge": comment_df["any_gauge_present"],
    "comment + health_comment": comment_df["health_comment_present"],
    "comment + tags + any_gauge": comment_df["tags_present"] & comment_df["any_gauge_present"],
    "comment only (no tags, no gauge)": (~comment_df["tags_present"]) & (~comment_df["any_gauge_present"]),
}

cooccurrence_summary = pd.DataFrame([
    {
        "condition": name,
        "n": int(mask.sum()),
        "pct_of_comments": round(mask.mean() * 100, 2),
    }
    for name, mask in cooccurrence_items.items()
])

display(cooccurrence_summary)

,condition,n,pct_of_comments
0,comment + tags,288,75.59
1,comment + any_gauge,363,95.28
2,comment + health_comment,226,59.32
3,comment + tags + any_gauge,288,75.59
4,"comment only (no tags, no gauge)",18,4.72


## 7. Comment가 tags / `special_mark`에 없는 추가 evidence를 주는지 확인

각 feature에 대해 comment에서 탐색된 keyword hit와 tags / special_mark 대비 **evidence candidate** 수를 봅니다. 모든 수치는 keyword screening이며 실제 행동 evidence coverage를 의미하지 않습니다.

이 역시 screening이며 실제 label 정확도는 수동 검토가 필요합니다.

In [39]:
FEATURE_PATTERNS = {
    "activity_energy": r"활동|활발|산책|운동|에너지|뛰어|뛰놀|놀이|놀기",
    "human_sociability": r"사람|낯선\s*사람|애교|친화|경계|겁이|겁많|무서|손길",
    "dog_compatibility": r"다른\s*(?:개|강아지)|강아지와|강아지랑|개와|개랑|반려견|동족",
    "cat_compatibility": r"다른\s*고양이|고양이와|고양이랑|냥이와|냥이랑|묘와|묘랑",
    "kids_compatibility": r"어린아이|아동|유아|아이들과|아이와\s*(?:잘|함께)|아이\s*있는\s*집|아기와",
    "housetraining": r"배변|실외배변|실내배변|패드|화장실",
    "owner_experience": r"초보\s*보호자|초보자|경험자|경험이\s*있는|처음\s*키우",
    "special_care": r"치료|투약|약을|질환|수술|장애|심장|피부|알레르기|재활|관리\s*필요",
    "alone_time": r"혼자\s*있는|혼자\s*두|분리불안|상주\s*인원",
    "training_need": r"교육\s*필요|훈련|사회화|입질\s*교육|적응\s*훈련",
}

feature_rows = []

for feature, pattern in FEATURE_PATTERNS.items():
    comment_hit = comment_df["personality_comment"].str.contains(
        pattern, regex=True, case=False, na=False
    )
    tags_hit = comment_df["personality_tags"].str.contains(
        pattern, regex=True, case=False, na=False
    )
    special_hit = comment_df["special_mark"].str.contains(
        pattern, regex=True, case=False, na=False
    )

    feature_rows.append({
        "feature": feature,
        "comment_n": len(comment_df),
        "comment_hit_n": int(comment_hit.sum()),
        "comment_hit_pct": round(comment_hit.mean() * 100, 2),
        "tags_hit_n": int(tags_hit.sum()),
        "special_mark_hit_n": int(special_hit.sum()),
        "comment_only_vs_tags_n": int((comment_hit & ~tags_hit).sum()),
        "comment_only_vs_special_mark_n": int((comment_hit & ~special_hit).sum()),
        "comment_only_vs_both_n": int((comment_hit & ~tags_hit & ~special_hit).sum()),
    })

comment_feature_screening = (
    pd.DataFrame(feature_rows)
      .sort_values(["comment_hit_n", "feature"], ascending=[False, True])
)

display(comment_feature_screening)

,feature,comment_n,comment_hit_n,comment_hit_pct,tags_hit_n,special_mark_hit_n,comment_only_vs_tags_n,comment_only_vs_special_mark_n,comment_only_vs_both_n
1,human_sociability,381,276,72.44,267,114,57,199,29
0,activity_energy,381,151,39.63,198,18,40,136,36
9,training_need,381,56,14.70,20,1,46,56,46
2,dog_compatibility,381,48,12.60,122,0,26,48,26
7,special_care,381,30,7.87,0,17,30,26,26
8,alone_time,381,24,6.30,2,2,24,24,24
5,housetraining,381,15,3.94,19,0,9,15,9
6,owner_experience,381,6,1.57,0,0,6,6,6
4,kids_compatibility,381,4,1.05,2,0,4,4,4
3,cat_compatibility,381,3,0.79,19,0,2,3,2


## 8. Comment 본문의 source / 작성주체 단서

이 단계는 **작성자를 확정하는 분석이 아닙니다.**

본문에서 포인핸드, 보호센터, 입양문의 등의 표현이 얼마나 등장하는지만 확인합니다.

In [40]:
SOURCE_MARKER_PATTERNS = {
    "mentions_pawinhand": r"포인핸드|pawinhand",
    "mentions_shelter_or_center": r"보호소|동물보호센터|동물복지지원센터|구조동물|센터\s*품",
    "mentions_adoption_contact": r"입양\s*문의|문의\s*[:：]?\s*\d|📞|☎|연중무휴",
    "mentions_socialization_program": r"사회화\s*프로그램",
    "mentions_ideal_home": r"이상적인\s*가정|추천\s*가정|단독가정추천",
}

source_marker_rows = []

for marker, pattern in SOURCE_MARKER_PATTERNS.items():
    mask = comment_df["personality_comment"].str.contains(
        pattern, regex=True, case=False, na=False
    )

    source_marker_rows.append({
        "marker": marker,
        "n": int(mask.sum()),
        "pct_of_comments": round(mask.mean() * 100, 2),
    })

comment_source_marker_summary = pd.DataFrame(source_marker_rows)
display(comment_source_marker_summary)

# notice_id의 -P suffix도 의미를 추정하지 않고 사실만 집계한다.
comment_df["notice_id_P_suffix"] = (
    comment_df["notice_id"].fillna("").astype(str).str.endswith("-P")
)

p_suffix_summary = pd.DataFrame({
    "group": ["P_suffix", "non_P_suffix"],
    "n": [
        int(comment_df["notice_id_P_suffix"].sum()),
        int((~comment_df["notice_id_P_suffix"]).sum()),
    ],
})

p_suffix_summary["pct_of_comments"] = (
    p_suffix_summary["n"] / len(comment_df) * 100
).round(2)

display(p_suffix_summary)

,marker,n,pct_of_comments
0,mentions_pawinhand,12,3.15
1,mentions_shelter_or_center,118,30.97
2,mentions_adoption_contact,80,21.00
3,mentions_socialization_program,12,3.15
4,mentions_ideal_home,54,14.17


,group,n,pct_of_comments
0,P_suffix,71,18.64
1,non_P_suffix,310,81.36


## 9. 반복 템플릿 / 중복 가능성

공백 정규화 후 완전히 동일한 comment와 반복되는 앞 80자를 기초적으로 확인합니다.

In [41]:
def normalize_comment(text):
    text = str(text)
    return re.sub(r"\s+", " ", text).strip().lower()


comment_df["comment_norm"] = comment_df["personality_comment"].map(normalize_comment)
comment_df["comment_prefix80"] = comment_df["comment_norm"].str[:80]

exact_counts = comment_df["comment_norm"].value_counts()
prefix_counts = comment_df["comment_prefix80"].value_counts()

template_summary = pd.DataFrame({
    "metric": [
        "comment_n",
        "unique_normalized_comments",
        "rows_in_exact_duplicate_groups",
        "exact_duplicate_group_n",
        "rows_with_repeated_prefix80",
        "repeated_prefix80_group_n",
    ],
    "value": [
        len(comment_df),
        int(comment_df["comment_norm"].nunique()),
        int(exact_counts[exact_counts > 1].sum()),
        int((exact_counts > 1).sum()),
        int(prefix_counts[prefix_counts > 1].sum()),
        int((prefix_counts > 1).sum()),
    ],
})

display(template_summary)

print("\n반복 prefix 상위")
display(
    prefix_counts[prefix_counts > 1]
    .head(20)
    .rename_axis("prefix80")
    .reset_index(name="n")
)

,metric,value
0,comment_n,381
1,unique_normalized_comments,334
2,rows_in_exact_duplicate_groups,58
3,exact_duplicate_group_n,11
4,rows_with_repeated_prefix80,62
5,repeated_prefix80_group_n,13



반복 prefix 상위


,prefix80,n
0,[양평군 유기동물보호센터 품] 📞입양문의 031-770-2311 연중무휴 09:00~18:00(점심시간 12:00~13:00),25
1,"안녕하세요 상주시 동물보호센터입니다˘ᗜ˘ 이 친구는 공고기한이 지난 8월 3일 부터 입양이 가능합니다. 보호소 운영시간은 9-18시 이나, 보호",6
2,*더 많은 사진은 인스타그램 jinju_dogs 참고해주세요^^,5
3,"*** 제2보호센터에서 보호중임 ( 제주시 어림비로616-107 , 064-710-4805~7 )",5
4,긴급구조 민원 관련하여 절차가 마무리 될 때까지 신청서 접수를 받지 않습니다.,4
5,"6마리 남매들 중 하얀 털을 가진 ♥삼공주♥를 소개합니다 마냥 하얘보이지만 자세히 보면 갈색의 땡땡이 무늬가 돋보이는 첫째(공고번호 434, 주",3
6,사람을 좋아하고 애교가 많음.,2
7,[신청자 다수로 신청서를 받지 않습니다],2
8,"○ 성격 - 사람 친화력 : 상, 사람 손길을 좋아하며 친화적으로 다가와요 - 타 동물 친화력 : 중, 익숙한 친구들과는 잘 어울려 놀아요 -",2
9,"○ 성격 - 사람 친화력 : 상, 사람 손길을 좋아하며 친화적으로 다가와요 - 타 동물 친화력 : 상, 강아지 친구들과 활발하게 어울려 놀아요",2


## 10. 보호센터별 comment 분포와 keyword screening

`shelter_name`별 전체 record와 `personality.comment` record를 집계합니다. 구조화 후보는 지정된 여섯 표현 중 하나 이상의 keyword hit로 정의하며, 실제 표준 template이라고 단정하지 않습니다.


In [42]:
STRUCTURED_COMMENT_PATTERN = (
    r"사람\s*친화력|타\s*동물\s*친화력|에너지\s*(?:레벨|수준)|"
    r"좋아하는\s*것|싫어하는\s*것|이상적인\s*가정"
)

shelter_df = df.copy()
shelter_df["shelter_name"] = shelter_df["shelter_name"].fillna("(결측)").replace("", "(결측)")
shelter_df["structured_comment_candidate"] = (
    shelter_df["personality_comment"].str.contains(
        STRUCTURED_COMMENT_PATTERN, regex=True, case=False, na=False
    ) & shelter_df["comment_present"]
)
shelter_comment_summary = (
    shelter_df.groupby("shelter_name", dropna=False)
    .agg(
        total_n=("notice_id", "size"),
        comment_n=("comment_present", "sum"),
        structured_comment_n=("structured_comment_candidate", "sum"),
    )
    .reset_index()
)
shelter_comment_summary["comment_coverage_pct"] = (
    shelter_comment_summary["comment_n"] / shelter_comment_summary["total_n"] * 100
).round(2)
shelter_comment_summary["structured_comment_pct"] = (
    shelter_comment_summary["structured_comment_n"] / shelter_comment_summary["comment_n"].replace(0, pd.NA) * 100
).fillna(0).round(2)
shelter_comment_summary = shelter_comment_summary.sort_values(
    ["comment_n", "shelter_name"], ascending=[False, True]
)

comment_shelter_n = int((shelter_comment_summary["comment_n"] > 0).sum())
shelter_n = int(len(shelter_comment_summary))
shelters_ge5_n = int((shelter_comment_summary["comment_n"] >= 5).sum())
shelters_ge10_n = int((shelter_comment_summary["comment_n"] >= 10).sum())
top1_share = round(shelter_comment_summary.head(1)["comment_n"].sum() / len(comment_df) * 100, 2)
top5_share = round(shelter_comment_summary.head(5)["comment_n"].sum() / len(comment_df) * 100, 2)
top10_share = round(shelter_comment_summary.head(10)["comment_n"].sum() / len(comment_df) * 100, 2)
structured_shelter_summary = shelter_comment_summary[
    shelter_comment_summary["structured_comment_n"] > 0
][["shelter_name", "structured_comment_n"]].copy()

display(shelter_comment_summary.head(20))
print({
    "shelter_n": shelter_n, "comment_shelter_n": comment_shelter_n,
    "shelters_ge5_n": shelters_ge5_n, "shelters_ge10_n": shelters_ge10_n,
    "top1_share": top1_share, "top5_share": top5_share, "top10_share": top10_share,
})
display(structured_shelter_summary)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_12860\2265590224.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0).round(2)


,shelter_name,total_n,comment_n,structured_comment_n,comment_coverage_pct,structured_comment_pct
140,양평군유기동물보호소,115,73,0,63.48,0.0
187,제주 동물보호센터,550,58,0,10.55,0.0
110,상주시 동물보호센터,70,31,0,44.29,0.0
112,서울시립동물복지지원센터 동대문,28,28,28,100.00,100.0
113,서울시립동물복지지원센터 마포,27,27,27,100.00,100.0
197,진주시동물보호센터,116,27,0,23.28,0.0
45,남양주시동물보호센터,158,21,0,13.29,0.0
105,사천시 동물보호센터,108,21,0,19.44,0.0
109,삼척시동물보호센터,45,18,0,40.00,0.0
9,가평군유기동물보호소,42,16,0,38.10,0.0


{'shelter_n': 245, 'comment_shelter_n': 19, 'shelters_ge5_n': 15, 'shelters_ge10_n': 12, 'top1_share': np.float64(19.16), 'top5_share': np.float64(56.96), 'top10_share': np.float64(83.99)}


,shelter_name,structured_comment_n
112,서울시립동물복지지원센터 동대문,28
113,서울시립동물복지지원센터 마포,27


## 11. 실제 comment 예시 검토

`STRUCTURED_ONLY = True`로 바꾸면 `사람 친화력`, `타 동물 친화력`, `에너지 레벨`, `이상적인 가정` 같은 섹션이 있는 comment를 중심으로 봅니다.

In [43]:
STRUCTURED_ONLY = False
N_EXAMPLES = 20

structured_mask = comment_df["personality_comment"].str.contains(
    r"사람\s*친화력|타\s*동물\s*친화력|에너지\s*(?:레벨|수준)|"
    r"좋아하는\s*것|싫어하는\s*것|이상적인\s*가정",
    regex=True,
    case=False,
    na=False,
)

example_pool = comment_df[structured_mask] if STRUCTURED_ONLY else comment_df

examples = example_pool[
    [
        "notice_id",
        "species",
        "status_raw",
        "personality_comment",
        "personality_tags",
        "gauge_활동성",
        "gauge_사회성",
        "gauge_친화도",
        "special_mark",
    ]
].sample(
    n=min(N_EXAMPLES, len(example_pool)),
    random_state=42,
)

display(examples)

,notice_id,species,status_raw,personality_comment,personality_tags,gauge_활동성,gauge_사회성,gauge_친화도,special_mark
15924,제주-제주-2026-01788,개,종료(입양),"엄마 없이 남매들만 보호소에 들어왔어요.\n지금 불린사료 먹으면서 하루하루 잘 크고 있는 귀염둥이들\n아직 1kg 남짓한 아기 강아지인데도 제법 의젓해요!\n잘먹고 잘 놀고, 사회성도 좋아요.\n매일매일 귀엽고 사랑스러움을 더해가는 아기 강아지의 평생 가족이 되어주세요~!!\n\n1.3kg (8/11기준)",,4,4,4,개체관리번호 1459 - 2주차
12978,경남-진주-2026-00261,개,보호중,"초롱초롱한 눈망울로 세상을 바라보는, 1~2살 된 시바견을 소개합니다. \n시바견 특유의 매력적인 외모는 물론, 이름처럼 따뜻하고 밝은 에너지를 가진 친구입니다. \n가장 큰 매력은 천사 같은 성격입니다. \n낯선 사람에게도 꼬리를 살랑살랑 흔들며 다가와 품에 폭 안기는 순둥이 중의 순둥이예요.\n\n아직 세상 모든 게 신기하고 재미있는 호기심쟁이라, 산책 나가는 걸 제일 좋아해요. ...",😘 애교많음 | 🥰 사람좋아함 | 🙂 활동량보통 | 👀 호기심많음,4,4,4,순하고 착함
3225,경기-양평-2026-00265,개,보호중,[양평군 유기동물보호센터 품]\n📞입양문의 031-770-2311\n연중무휴 09:00~18:00(점심시간 12:00~13:00),,4,3,3,"겁이 많고 경계심 높음, 주황색 목걸이 착용"
4613,경기-양평-2026-00256,개,보호중,[양평군 유기동물보호센터 품]\n📞입양문의 031-770-2311\n연중무휴 09:00~18:00(점심시간 12:00~13:00),🙈 낯가림있음 | 😟 겁이많음 | 🐾 산책좋아함 | 🧘 차분한성격,4,4,3,사람에게 경계심은 있지만 공격성은 없고 온순한 성격
4615,경기-양평-2026-00254,개,종료(입양),[입양확정으로 신청서를 받지않습니다]\n🤍 호기심 가득한 아기 강아지 🤍\n\n🔎 특징\n-사람 손에서 자라 사람을 정말 좋아하는 아이예요. 🥰\n-앙증맞은 외모와 살짝 접힌 수제비 귀가 매력적😍.\n-쌍커풀 있는 듯한 큰 눈.😄\n\n😄 성격·성향\n-호기심이 많고 활발해요.😁\n-작은 몸에 비해 점프력도 좋아요👍\n-자기주장을 할 줄 아는 아이에요. 😆\n\n❗ 특이사항\n-어린 ...,🥰 사람좋아함 | ⚡ 활동량많음 | 👀 호기심많음 | 📚 교육필요,5,5,5,7월출생/활발하고 활동성높음 사람을 좋아함.
2829,서울-시립1-2026-00094-P,개,보호중,"<입양 신청 가능합니다.>\n\n모두 나와 친구하자~ 활기 넘치는 하룻강아지, \n ""재미"" - 3개월령 (2026년 06월생 추정) \n\n​\n[성격]\n• 사람 친화력 : 상 / 성급하게 다가가지 않는다면 먼저 다가와서 인사하는 순하고 착한 강아지\n• 타 동물 친화력 : 상 / 다른 강아지들에게 먼저 놀자고 하거나 인사하는 호감형 강아지 (고양이 경험은 없습...",🥰 사람좋아함 | 🐶 다른개와잘지냄 | ⚡ 활동량많음 | 👀 호기심많음 | 📚 교육필요,5,5,5,"재미 / 모두 나와 친구하자~ 활기 넘치는 하룻강아지, ""재미"""
10824,서울-시립2-2026-00052-P,개,보호중,"○ 성격 \n- 사람 친화력 : 상, 사람 손길을 좋아하며 친화적으로 다가와요\n- 타 동물 친화력 : 중, 사람과 가져와 놀이 하는걸 좋아해요, 장난감 소유욕이 있는편이에요\n- 에너지 레벨 : 중\n- 좋아하는 것 : 쓰담쓰담 손길, 사람과 공놀이\n- 싫어하는 것 : 갖고있던 물건(장난감) 뺏기, 갑자기 만지기, 연령대가 높은 할머니, 할아버지\n\n○ 이상적인 가정 \n- 강아...",🥰 사람좋아함 | 😘 애교많음 | ⚡ 활동량많음 | 🎾 놀이좋아함 | 🏠 단독가정추천,5,4,5,캐치볼을 사랑하는 강아지 ‘땡칠’ 입니다.
21615,경기-남양주-2026-00259,개,보호중,"동그란 눈망울과 쫑긋 선 귀가 너무나 사랑스러운 아이예요.🐕‍🦺🐾\n\n하얀 크림빛 털에 귀여운 얼굴 무늬까지 , 보면 볼수록 매력이 넘치는 친구입니다.\n\n잠시 예뻐해주는 가족이 아니라 아플 때도 곁에 있고 기쁠 때 함께 웃으며\n\n평생 가족이 되어주실 분을 기다립니다💕",🥰 사람좋아함 | 🐶 다른개와잘지냄 | 🙂 활동량보통 | 🐕 산책가능,4,4,4,겁이 많지만 사람의 손길을 피하지 않음
13635,경기-양평-2026-00214,개,종료(반환),🐶친칠라를 닮은 듯한 멈무이🐶\n\n🔎특징\n-양쪽 귀의 털이 날개를 펼친듯한 외형이라 신비롭고 매력적\n(색의 조화도 'so good' 하답니다~)\n-빠져들 듯한 눈으로 아이컨택을 잘 함\n\n😄성격/성향\n-사람을 좋아하고 만져달라고 애교를 부리는 편\n-차분한 성격이기도 하지만 조금은 소심한 모습이 있음\n(친구들이 사람 주변에 많으면 파고들지 못하고 쭈뼛거리며 주변에 앉아서 ...,,4,4,3,"경계심이 살짝 있지만 사람에게 공격성은 전혀 없음, 임신한 것으로 추정됨"
7036,경남-진주-2026-00303,개,보호중,*더 많은 사진은 인스타그램 jinju_dogs 참고해주세요^^,🙈 낯가림있음 | 😟 겁이많음 | 🧘 차분한성격 | 👂 경계심있음,2,2,2,겁많고 방어적 (엄살끼 있음)


## 12. Comment 수동 review sample

아래 `manual_*` 칼럼은 **이 notebook이 새로 만드는 검토용 칼럼**이며 PawInHand 원본 데이터가 아닙니다.

- `manual_author_source`: 작성주체에 대한 사람이 확인한 메모. 확정 불가면 `unknown`
- `manual_behavior_evidence`: 행동/생활 feature 근거가 실질적으로 있는지
- `manual_template_like`: 템플릿/홍보성 문구가 강한지
- `manual_evidence_span`: 판단 근거 원문
- `manual_review_notes`: 기타 메모

In [44]:
COMMENT_REVIEW_N = 120

comment_review_sample = comment_df.sample(
    n=min(COMMENT_REVIEW_N, len(comment_df)),
    random_state=42,
)[
    [
        "notice_id",
        "species",
        "breed",
        "status_raw",
        "personality_comment",
        "personality_tags",
        "gauge_건강상태",
        "gauge_활동성",
        "gauge_사회성",
        "gauge_친화도",
        "special_mark",
        "comment_char_len",
        "notice_id_P_suffix",
    ]
].copy()

comment_review_sample["manual_author_source"] = ""
comment_review_sample["manual_behavior_evidence"] = ""
comment_review_sample["manual_template_like"] = ""
comment_review_sample["manual_evidence_span"] = ""
comment_review_sample["manual_review_notes"] = ""

display(comment_review_sample.head(10))

,notice_id,species,breed,status_raw,personality_comment,personality_tags,gauge_건강상태,gauge_활동성,gauge_사회성,gauge_친화도,special_mark,comment_char_len,notice_id_P_suffix,manual_author_source,manual_behavior_evidence,manual_template_like,manual_evidence_span,manual_review_notes
15924,제주-제주-2026-01788,개,[개] 믹스견,종료(입양),"엄마 없이 남매들만 보호소에 들어왔어요.\n지금 불린사료 먹으면서 하루하루 잘 크고 있는 귀염둥이들\n아직 1kg 남짓한 아기 강아지인데도 제법 의젓해요!\n잘먹고 잘 놀고, 사회성도 좋아요.\n매일매일 귀엽고 사랑스러움을 더해가는 아기 강아지의 평생 가족이 되어주세요~!!\n\n1.3kg (8/11기준)",,4,4,4,4,개체관리번호 1459 - 2주차,166,False,,,,,
12978,경남-진주-2026-00261,개,[개] 시바,보호중,"초롱초롱한 눈망울로 세상을 바라보는, 1~2살 된 시바견을 소개합니다. \n시바견 특유의 매력적인 외모는 물론, 이름처럼 따뜻하고 밝은 에너지를 가진 친구입니다. \n가장 큰 매력은 천사 같은 성격입니다. \n낯선 사람에게도 꼬리를 살랑살랑 흔들며 다가와 품에 폭 안기는 순둥이 중의 순둥이예요.\n\n아직 세상 모든 게 신기하고 재미있는 호기심쟁이라, 산책 나가는 걸 제일 좋아해요. ...",😘 애교많음 | 🥰 사람좋아함 | 🙂 활동량보통 | 👀 호기심많음,4,4,4,4,순하고 착함,395,False,,,,,
3225,경기-양평-2026-00265,개,[개] 믹스견,보호중,[양평군 유기동물보호센터 품]\n📞입양문의 031-770-2311\n연중무휴 09:00~18:00(점심시간 12:00~13:00),,5,4,3,3,"겁이 많고 경계심 높음, 주황색 목걸이 착용",70,False,,,,,
4613,경기-양평-2026-00256,개,[개] 믹스견,보호중,[양평군 유기동물보호센터 품]\n📞입양문의 031-770-2311\n연중무휴 09:00~18:00(점심시간 12:00~13:00),🙈 낯가림있음 | 😟 겁이많음 | 🐾 산책좋아함 | 🧘 차분한성격,5,4,4,3,사람에게 경계심은 있지만 공격성은 없고 온순한 성격,70,False,,,,,
4615,경기-양평-2026-00254,개,[개] 믹스견,종료(입양),[입양확정으로 신청서를 받지않습니다]\n🤍 호기심 가득한 아기 강아지 🤍\n\n🔎 특징\n-사람 손에서 자라 사람을 정말 좋아하는 아이예요. 🥰\n-앙증맞은 외모와 살짝 접힌 수제비 귀가 매력적😍.\n-쌍커풀 있는 듯한 큰 눈.😄\n\n😄 성격·성향\n-호기심이 많고 활발해요.😁\n-작은 몸에 비해 점프력도 좋아요👍\n-자기주장을 할 줄 아는 아이에요. 😆\n\n❗ 특이사항\n-어린 ...,🥰 사람좋아함 | ⚡ 활동량많음 | 👀 호기심많음 | 📚 교육필요,5,5,5,5,7월출생/활발하고 활동성높음 사람을 좋아함.,318,False,,,,,
2829,서울-시립1-2026-00094-P,개,[개] 믹스견,보호중,"<입양 신청 가능합니다.>\n\n모두 나와 친구하자~ 활기 넘치는 하룻강아지, \n ""재미"" - 3개월령 (2026년 06월생 추정) \n\n​\n[성격]\n• 사람 친화력 : 상 / 성급하게 다가가지 않는다면 먼저 다가와서 인사하는 순하고 착한 강아지\n• 타 동물 친화력 : 상 / 다른 강아지들에게 먼저 놀자고 하거나 인사하는 호감형 강아지 (고양이 경험은 없습...",🥰 사람좋아함 | 🐶 다른개와잘지냄 | ⚡ 활동량많음 | 👀 호기심많음 | 📚 교육필요,5,5,5,5,"재미 / 모두 나와 친구하자~ 활기 넘치는 하룻강아지, ""재미""",485,True,,,,,
10824,서울-시립2-2026-00052-P,개,[개] 토이푸들,보호중,"○ 성격 \n- 사람 친화력 : 상, 사람 손길을 좋아하며 친화적으로 다가와요\n- 타 동물 친화력 : 중, 사람과 가져와 놀이 하는걸 좋아해요, 장난감 소유욕이 있는편이에요\n- 에너지 레벨 : 중\n- 좋아하는 것 : 쓰담쓰담 손길, 사람과 공놀이\n- 싫어하는 것 : 갖고있던 물건(장난감) 뺏기, 갑자기 만지기, 연령대가 높은 할머니, 할아버지\n\n○ 이상적인 가정 \n- 강아...",🥰 사람좋아함 | 😘 애교많음 | ⚡ 활동량많음 | 🎾 놀이좋아함 | 🏠 단독가정추천,5,5,4,5,캐치볼을 사랑하는 강아지 ‘땡칠’ 입니다.,301,True,,,,,
21615,경기-남양주-2026-00259,개,[개] 믹스견,보호중,"동그란 눈망울과 쫑긋 선 귀가 너무나 사랑스러운 아이예요.🐕‍🦺🐾\n\n하얀 크림빛 털에 귀여운 얼굴 무늬까지 , 보면 볼수록 매력이 넘치는 친구입니다.\n\n잠시 예뻐해주는 가족이 아니라 아플 때도 곁에 있고 기쁠 때 함께 웃으며\n\n평생 가족이 되어주실 분을 기다립니다💕",🥰 사람좋아함 | 🐶 다른개와잘지냄 | 🙂 활동량보통 | 🐕 산책가능,5,4,4,4,겁이 많지만 사람의 손길을 피하지 않음,148,False,,,,,
13635,경기-양평-2026-00214,개,[개] 믹스견,종료(반환),🐶친칠라를 닮은 듯한 멈무이🐶\n\n🔎특징\n-양쪽 귀의 털이 날개를 펼친듯한 외형이라 신비롭고 매력적\n(색의 조화도 'so good' 하답니다~)\n-빠져들 듯한 눈으로 아이컨택을 잘 함\n\n😄성격/성향\n-사람을 좋아하고 만져달라고 애교를 부리는 편\n-차분한 성격이기도 하지만 조금은 소심한 모습이 있음\n(친구들이 사람 주변에 많으면 파고들지 못하고 쭈뼛거리며 주변에 앉아서 ...,,4,4,4,3,"경계심이 살짝 있지만 사람에게 공격성은 전혀 없음, 임신한 것으로 추정됨",322,False,,,,,
7036,경남-진주-2026-00303,개,[개] 믹스견,보호중,*더 많은 사진은 인스타그램 jinju_dogs 참고해주세요^^,🙈 낯가림있음 | 😟 겁이많음 | 🧘 차분한성격 | 👂 경계심있음,4,2,2,2,겁많고 방어적 (엄살끼 있음),35,False,,,,,


## 13. 최종 산출물 저장

In [45]:
# 정해진 구버전 CSV만 제거합니다. 디렉터리 전체 삭제는 하지 않습니다.
OBSOLETE_RESULT_FILES = [
    "source_coverage_summary.csv", "species_coverage_summary.csv",
    "gauge_distribution.csv", "tag_distribution.csv",
    "keyword_screening_summary.csv", "representative_review_sample.csv",
    "enriched_review_sample.csv", "feature_keyword_review_sample.csv",
    "comment_by_species.csv", "comment_by_status.csv",
    "comment_by_species_status.csv", "comment_length_summary.csv",
    "comment_length_distribution.csv", "comment_structure_summary.csv",
    "comment_cooccurrence_summary.csv", "comment_source_marker_summary.csv",
    "comment_notice_id_suffix_summary.csv", "comment_template_summary.csv",
    "raw_schema_path_summary.csv", "raw_source_field_candidates.csv",
    "comment_feature_screening_summary.csv",
]

summary_rows = []
def add_summary(category, metric, group="all", n=None, pct=None, value=None):
    summary_rows.append({"category": category, "metric": metric, "group": group, "n": n, "pct": pct, "value": value})

for _, r in coverage_summary.iterrows():
    metric = "detail_present_clean" if r["source"] == "detail_present_clean" else r["source"]
    add_summary("coverage", metric, r["scope"], int(r["non_null_n"]), float(r["coverage_pct"]), int(r["n_total"]))
for label, frame, key in [("species", comment_by_species, "species"), ("status", comment_by_status, "status_raw")]:
    for _, r in frame.iterrows():
        add_summary("comment_coverage", key, str(r[key]), int(r["comment_n"]), float(r["comment_pct"]), int(r["n"]))
for _, r in length_summary.iterrows():
    add_summary("comment_length", str(r["metric"]), "comment", value=r["value"])
for _, r in comment_structure_summary.iterrows():
    add_summary("topic_keyword_screening", str(r["section_or_topic"]), "comment", int(r["hit_n"]), float(r["pct_of_comments"]))
for _, r in cooccurrence_summary.iterrows():
    add_summary("cooccurrence", str(r["condition"]), "comment", int(r["n"]), float(r["pct_of_comments"]))
for _, r in comment_source_marker_summary.iterrows():
    add_summary("source_marker_keyword_screening", str(r["marker"]), "comment", int(r["n"]), float(r["pct_of_comments"]))
for _, r in template_summary.iterrows():
    add_summary("template_duplicate", str(r["metric"]), "comment", value=r["value"])
for metric, value in [("shelter_n", shelter_n), ("comment_shelter_n", comment_shelter_n), ("shelters_ge5_n", shelters_ge5_n), ("shelters_ge10_n", shelters_ge10_n)]:
    add_summary("shelter_concentration", metric, "all", n=value)
for metric, value in [("top1_share", top1_share), ("top5_share", top5_share), ("top10_share", top10_share)]:
    add_summary("shelter_concentration", metric, "comment", pct=value)
for _, r in structured_shelter_summary.iterrows():
    add_summary("structured_comment_keyword_screening", "structured_comment_n", str(r["shelter_name"]), int(r["structured_comment_n"]))

comment_feature_summary = comment_feature_screening.rename(columns={
    "comment_n": "comment_total_n",
}).copy()

# Write only the four final CSVs. The report is maintained alongside these files.
for filename in OBSOLETE_RESULT_FILES:
    old_path = OUTPUT_DIR / filename
    if old_path.exists():
        old_path.unlink()

outputs = {
    "audit_summary.csv": pd.DataFrame(summary_rows, columns=["category", "metric", "group", "n", "pct", "value"]),
    "comment_feature_summary.csv": comment_feature_summary,
    "shelter_comment_summary.csv": shelter_comment_summary[[
        "shelter_name", "total_n", "comment_n", "comment_coverage_pct",
        "structured_comment_n", "structured_comment_pct",
    ]],
    "comment_review_sample.csv": comment_review_sample,
}
for filename, frame in outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")
print("saved:", *outputs.keys(), sep="\n- ")


saved:
- audit_summary.csv
- comment_feature_summary.csv
- shelter_comment_summary.csv
- comment_review_sample.csv


## 14. 현재 분석 질문에 대한 답

### A. `personality.comment`가 실제 추가 정보를 주는가?

**현재 답: 제한적으로 Yes.** comment는 전체 22,511건 중 381건(1.69%)으로 coverage가 낮다. 반면 comment가 존재하는 record 안에서는 tags 또는 `special_mark`에 없는 추가 keyword evidence candidate가 확인된다. 교육 필요(46건), 혼자 있는 시간(24건), 특수 돌봄(26건), 활동성(36건), 개와의 관계(26건)에서 comment-only-vs-both screening hit가 있었다. 이는 keyword screening 결과이며 실제 행동 evidence coverage를 뜻하지 않는다. comment는 전체 profile을 채우는 주 source가 아니라 일부 개체에 richer evidence 후보를 제공하는 보조 source로 보는 편이 적절하다.

### B. comment의 구조가 반복 가능한가?

**현재 답: 일부 comment에서는 Yes, 전체적으로는 No.** 381건 중 사람 친화력 관련 표현 105건, 좋아하는 것 87건, 에너지/활동성 65건, 타 동물 친화력 60건, 교육/훈련 56건, 싫어하는 것 55건, 이상적인 가정 54건이 keyword screening에서 탐색됐다. 자유서술·기관 안내·연락처·홍보 문구가 섞여 있으므로 전체를 동일 template의 성격 프로필로 볼 수 없다.

### C. comment가 특정 기관/프로그램에 편중되어 있는가?

**현재 답: 일부 보호센터에 집중되어 있다.** 원본에서 shelter_name은 245개였고 comment 보유 shelter는 19개, 5건 이상은 15개, 10건 이상은 12개였다. 상위 1개 shelter가 comment의 19.16%(73/381), 상위 5개가 56.96%(217/381), 상위 10개가 83.99%(320/381)를 차지했다. 지정 표현 기반 structured comment candidate는 55건이며 서울시립동물복지지원센터 동대문 28건, 마포 27건으로 탐색됐다. 이 집중은 특정 기관의 기록 방식 영향 가능성을 보여주지만 작성 주체나 표준 template임을 확정하지 않는다.

### D. comment를 동물 성격의 ground truth로 사용할 수 있는가?

**현재 답: No.** 작성자와 기관별 평가 기준, 관찰 시점·기간·상황, 보호소 밖 환경에서의 행동 지속성, 평가자 간 reliability가 확인되지 않았다. comment는 contextual behavioral evidence 후보로만 다루며 고정된 true personality label로 사용하지 않는다.

### E. 현재 데이터만으로 추천 시스템을 객관적으로 평가할 수 있는가?

**현재 답: 아직 No.** user-animal interaction, 사용자별 relevance, 찜·문의·방문, 입양 후 만족도와 장기 유지 결과가 연결되어 있지 않다. extraction screening과 recommendation ranking 평가는 구분해야 하며 추천 evaluation은 unresolved issue로 남긴다.
